In [1]:
import numpy as np
import pandas as pd

In [2]:
fold0 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold0_with_positions_steps_results.tsv", sep="\t")

fold0["fold"] = [0 for i in range(len(fold0))]

In [3]:
fold1 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold1_with_positions_steps_results.tsv", sep="\t")

fold1["fold"] = [1 for i in range(len(fold1))]

In [4]:
fold2 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold2_with_positions_steps_results.tsv", sep="\t")

fold2["fold"] = [2 for i in range(len(fold2))]

In [5]:
fold3 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold3_with_positions_steps_results.tsv", sep="\t")

fold3["fold"] = [3 for i in range(len(fold3))]

In [6]:
fold4 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold4_with_positions_steps_results.tsv", sep="\t")

fold4["fold"] = [4 for i in range(len(fold4))]

In [7]:
fold5 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold5_with_positions_steps_results.tsv", sep="\t")

fold5["fold"] = [5 for i in range(len(fold5))]

In [8]:
fold6 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold6_with_positions_steps_results.tsv", sep="\t")

fold6["fold"] = [6 for i in range(len(fold6))]

In [9]:
fold7 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold7_with_positions_steps_results.tsv", sep="\t")

fold7["fold"] = [7 for i in range(len(fold7))]

In [10]:
df = pd.concat([fold0, fold1, fold2,
                fold3, fold4, fold5,
                fold6, fold7], ignore_index=True)

In [11]:
len(df)

406

In [12]:
df["URQ_delta"] = df["URQ_result"] - df["URQ_init"]

In [13]:
# selecting only sequences with a measurable contact enrichment
df = df[df['URQ_delta'] > 0.0]

In [14]:
len(df)

389

### saving sequences to fasta

In [15]:
import torch
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from pyfaidx import Fasta

In [16]:
fasta_file = "/project/fudenber_735/genomes/mm10/mm10.fa"
sequence_dir = "/scratch1/smaruj/suppressing_CTCFs/results"
output_fasta = "combined_sequences.fasta"

In [17]:
genome = Fasta(fasta_file)

In [18]:
def ohe_to_seq(tensor):
    base_order = ['A', 'C', 'G', 'T']
    tensor = tensor.permute(1, 0)
    indices = torch.argmax(tensor, dim=1)
    return ''.join([base_order[i.item()] for i in indices])

In [19]:
bin_size = 2048
cropping_applied = 64
padding_bins = 2
padding = padding_bins * bin_size

slice_0_bins = [256]
slice_0_start = (min(slice_0_bins) + cropping_applied - padding_bins) * bin_size
slice_0_end = (max(slice_0_bins) + 1 + cropping_applied + padding_bins) * bin_size

In [20]:
records = []
for i, row in df.iterrows():
    chrom = row['chrom']
    centered_start = int(row['centered_start'])
    centered_end = int(row['centered_end'])
    start = centered_start + slice_0_start
    end = centered_start + slice_0_end
    fold = row['fold']

    # (1) Get 60 bp upstream
    upstream = genome[chrom][start - 60:start].seq.upper()

    # (2) Read .pt file and decode
    seq_path = f"{sequence_dir}/fold{fold}/{chrom}_{centered_start}_{centered_end}_slice.pt"
    ohe_tensor = torch.load(seq_path)  # shape: (2048, 4)
    seq_string = ohe_to_seq(ohe_tensor.squeeze(0))

    # (3) Get 60 bp downstream
    downstream = genome[chrom][end:end + 60].seq.upper()

    # (4) Combine all parts
    full_seq = upstream + seq_string + downstream

    # (5) Create FASTA record
    header = f"{chrom}_{centered_start}_{centered_end}"
    record = SeqRecord(Seq(full_seq), id=header, description="")
    records.append(record)

# with open(f"{sequence_dir}/{output_fasta}", "w") as out_f:
#     SeqIO.write(records, out_f, "fasta")

# print(f"✅ Done. {len(records)} sequences written to {output_fasta}")

/tmp/SLURM_1495880/ipykernel_2192657/3666545163.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ohe_tensor = torch.load(seq_path)  # shape: (2048, 4)


KeyboardInterrupt: 

### saving the original sequences

In [21]:
fasta_file = "/project/fudenber_735/genomes/mm10/mm10.fa"
sequence_dir = "/scratch1/smaruj/suppressing_CTCFs/results"
output_fasta = "OG_combined_sequences.fasta"

In [22]:
records = []
for i, row in df.iterrows():
    chrom = row['chrom']
    centered_start = int(row['centered_start'])
    centered_end = int(row['centered_end'])
    start = centered_start + slice_0_start
    end = centered_start + slice_0_end
    fold = row['fold']

    og_sequence = genome[chrom][start - 60:end + 60].seq.upper()

    # Create FASTA record
    header = f"{chrom}_{centered_start}_{centered_end}"
    record = SeqRecord(Seq(og_sequence), id=header, description="")
    records.append(record)

with open(f"{sequence_dir}/{output_fasta}", "w") as out_f:
    SeqIO.write(records, out_f, "fasta")

print(f"✅ Done. {len(records)} sequences written to {output_fasta}")

✅ Done. 389 sequences written to OG_combined_sequences.fasta


In [ ]:
# for i, row in df.iterrows():
#     chrom = row['chrom']
#     centered_start = int(row['centered_start'])
#     centered_end = int(row['centered_end'])
#     start = centered_start + slice_0_start
#     end = centered_start + slice_0_end
#     fold = row['fold']

#     og_sequence = genome[chrom][start - 60:end + 60].seq.upper()

#     # Create FASTA record
#     header = f"{chrom}_{centered_start}_{centered_end}"
#     record = SeqRecord(Seq(og_sequence), id=header, description="")

#     # (6) Write this sequence to its own FASTA file
#     out_path = f"{sequence_dir}/OG_seqs_fasta/{header}.fasta"
#     with open(out_path, "w") as out_f:
#         SeqIO.write(record, out_f, "fasta")

# print(f"✅ Done. {len(df)} separate FASTA files written to {sequence_dir}")

### Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df.columns

In [ ]:
# Set style
sns.set(style="whitegrid")

# Plot histogram with KDE
plt.figure(figsize=(8, 5))
sns.histplot(df['num_edits'], kde=False, bins=20, color='skyblue')
plt.title('Distribution of num_edits')
plt.xlabel('num_edits')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import ttest_rel

In [ ]:
t_stat, p_value = ttest_rel(df['URQ_result'], df['URQ_init'])
print(f"Paired t-test: t = {t_stat:.3f}, p = {p_value:.3e}")

In [ ]:
df_melted = df[['URQ_init', 'URQ_result']].melt(var_name='Metric', value_name='Value')

plt.figure(figsize=(8, 6))
sns.boxplot(x='Metric', y='Value', data=df_melted, palette='Set2')
plt.title(f'Boxplot of URQ_result vs URQ_init, p = {p_value:.3e}')
plt.ylabel('Value')
plt.xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
t_stat, p_value = ttest_rel(df['init_CTCFs_num'], df['CTCFs_num'])
print(f"Paired t-test: t = {t_stat:.3f}, p = {p_value:.3e}")

In [ ]:
df_melted = df[['init_CTCFs_num', 'CTCFs_num']].melt(var_name='Metric', value_name='Value')

plt.figure(figsize=(8, 6))
sns.boxplot(x='Metric', y='Value', data=df_melted, palette='Set2')
plt.title(f'Boxplot of init_CTCFs_num vs CTCFs_num, p = {p_value:.3e}')
plt.ylabel('Value')
plt.xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
t_stat, p_value = ttest_rel(df['GC_slice'], df['GC_slice_edited'])
print(f"Paired t-test: t = {t_stat:.3f}, p = {p_value:.3e}")

In [ ]:
# Reshape for seaborn boxplot
df_melted = df[['GC_slice', 'GC_slice_edited']].melt(var_name='Condition', value_name='Value')

plt.figure(figsize=(8, 5))
sns.boxplot(x='Condition', y='Value', data=df_melted, palette='pastel')
plt.title(f'Boxplot of GC_slice vs GC_slice_edited, p = {p_value:.3e}')
plt.xlabel('')
plt.ylabel('Value')
plt.tight_layout()
plt.show()

In [ ]:
df.columns

In [ ]:
t_stat, p_value = ttest_rel(df['avg_orig_fimo_scores'], df['avg_new_fimo_scores'])
print(f"Paired t-test: t = {t_stat:.3f}, p = {p_value:.3e}")

In [ ]:
# Reshape for seaborn boxplot
df_melted = df[['avg_orig_fimo_scores', 'avg_new_fimo_scores']].melt(var_name='Condition', value_name='Value')

plt.figure(figsize=(8, 5))
sns.boxplot(x='Condition', y='Value', data=df_melted, palette='pastel')
plt.title(f'Boxplot of avg_orig_fimo_scores vs avg_new_fimo_scores, p = {p_value:.3e}')
plt.xlabel('')
plt.ylabel('Value')
plt.tight_layout()
plt.show()